# m2 ASR: dev run for one model
DEV audio only (the common subset in `results/m2_asr/common_subset.txt`). The parameters cell is tagged for papermill.
Resumable: files already transcribed are skipped. Test audio is refused by `m2_asr.asr.guard`.
No number below is typed by hand: everything is read from `results/m2_asr/`.
`RUN = False` only re-reads saved results (the baseline was run from the command line in the background; its log is in the README).

In [ ]:
MODEL = "faster-whisper-small"  # parameters
THREADS = 8
LANGUAGE = "kn"
RUN = True

In [ ]:
import json, os, subprocess, sys, tempfile, platform
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "m2_asr").is_dir())
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
from m2_asr.asr import models

print("python", platform.python_version(), "| CPU:", models.cpu_name(), "| logical CPUs:", os.cpu_count())
print("free disk GB:", round(models.free_gb(), 2))
try:
    import torch

    print("torch CUDA available:", torch.cuda.is_available(), "(torch is CPU-only here; the CTranslate2 run uses the CPU)")
except Exception as e:
    print("torch:", e)
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip())

In [ ]:
scratch = Path(tempfile.gettempdir()) / "m2scratch"
if RUN:
    r = subprocess.run(
        [sys.executable, "-m", "m2_asr", "run", "--model", MODEL, "--language", LANGUAGE, "--threads", str(THREADS), "--scratch", str(scratch)],
        capture_output=True, text=True, encoding="utf-8",
    )
    print("\n".join(r.stdout.strip().splitlines()[-4:]))
    print("exit code", r.returncode)
run_json = json.loads((ROOT / "results/m2_asr" / MODEL / "run.json").read_text(encoding="utf-8"))
print({k: run_json[k] for k in ("model", "revision", "licence", "status", "hardware", "files_expected", "files_written", "failed", "wall_seconds")})
print("settings:", run_json["settings"])

In [ ]:
import csv

r = subprocess.run([sys.executable, "-m", "m2_asr", "summarise"], capture_output=True, text=True, encoding="utf-8")
print(r.stdout.strip() or r.stderr[-500:])
rows = list(csv.DictReader(open(ROOT / "results/m2_asr/summary.csv", encoding="utf-8")))


def g(x, k):
    return x.get(k, "")


hdr = f"{'source':12}{'slice':14}{'value':12}{'N':>5}  {'CER [95% boot CI]':26}{'gCER':>7}{'WER':>7}   neg   num  drug  unit  p50ms p95ms   RTF"
print(hdr)
for x in rows:
    if x["model"] != MODEL or x["status"] != "ran":
        continue
    if x["source"] == "public_real" and x["slice"] not in ("all", "code_switched"):
        continue
    if x["slice"] in ("noise_type", "snr_db", "speed") or x["slice"] == "voice" and x["source"] == "public_real":
        continue
    rec = " ".join((g(x, f"recall_{c}_k") + "/" + g(x, f"recall_{c}_n")).rjust(7) for c in ("negation", "number", "drug", "unit"))
    cer = f"{g(x,'cer_mean')} [{g(x,'cer_ci_lo')},{g(x,'cer_ci_hi')}]"
    print(f"{x['source']:12}{x['slice']:14}{x['value']:12}{x['n_utterances']:>5}  {cer:26}{g(x,'gcer_mean'):>7}{g(x,'wer_mean'):>7} {rec} {g(x,'latency_ms_p50'):>6} {g(x,'latency_ms_p95'):>6} {g(x,'rtf_mean'):>6}")

Reading guide: CER above 1.0 means the model wrote more characters than the reference (repetition loops).
The `neg / num / drug / unit` columns are hits/total reference terms (counts, not rates; Wilson CIs are in summary.csv).
TTS and public_real are never pooled. Clean TTS speech is NOT real patient speech. Latency = decode + inference per file, CPU only, GPU not used.